In [ ]:
import pandas as pd
import subprocess

# NCBI's official command-line tool for genome downloads
!curl -o datasets 'https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets'
!chmod +x datasets
!./datasets --version

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 19.0M  100 19.0M    0     0  32.8M      0 --:--:-- --:--:-- --:--:-- 32.8M
datasets version: 18.33.1


In [ ]:
# Get summary of all assemblies in the 902-isolate pharyngitis study — just metadata first
!./datasets summary genome accession PRJNA817826 --as-json-lines > bioproject_summary.jsonl

!wc -l bioproject_summary.jsonl
!head -n 2 bioproject_summary.jsonl

989 bioproject_summary.jsonl
{"accession":"GCF_053048175.1","annotation_info":{"method":"Best-placed reference protein set; GeneMarkS-2+","name":"GCF_053048175.1-RS_2025_11_17","pipeline":"NCBI Prokaryotic Genome Annotation Pipeline (PGAP)","provider":"NCBI RefSeq","release_date":"2025-11-17","software_version":"6.10","stats":{"gene_counts":{"non_coding":57,"protein_coding":1702,"pseudogene":88,"total":1847}}},"assembly_info":{"assembly_level":"Contig","assembly_method":"SPAdes v. 4.0.0","assembly_name":"ASM5304817v1","assembly_status":"current","assembly_type":"haploid","bioproject_accession":"PRJNA817826","bioproject_lineage":[{"bioprojects":[{"accession":"PRJNA817826","title":"Streptococcus pyogenes Genome sequencing"}]}],"biosample":{"accession":"SAMN52394264","attributes":[{"name":"strain","value":"GAS-191193"},{"name":"collected_by","value":"Unknown"},{"name":"collection_date","value":"2021"},{"name":"geo_loc_name","value":"USA:Georgia"},{"name":"host","value":"Homo sapiens"},{"n

In [ ]:
!git clone https://github.com/BenJamesMetcalf/GAS_Scripts_Reference.git
!ls GAS_Scripts_Reference/GAS_Reference_DB

Cloning into 'GAS_Scripts_Reference'...
remote: Enumerating objects: 334, done.
remote: Counting objects: 100% (109/109), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 334 (delta 44), reused 84 (delta 30), pack-reused 225 (from 1)
Receiving objects: 100% (334/334), 10.86 MiB | 9.39 MiB/s, done.
Resolving deltas: 100% (134/134), done.
alleles_fasta
ARGannot_r1.fasta
ARGannot_r1.fasta.1.bt2
ARGannot_r1.fasta.2.bt2
ARGannot_r1.fasta.3.bt2
ARGannot_r1.fasta.4.bt2
ARGannot_r1.fasta.fai
ARGannot_r1.fasta.rev.1.bt2
ARGannot_r1.fasta.rev.2.bt2
Blast_bLactam_2X_prot_DB.phr
Blast_bLactam_2X_prot_DB.pin
Blast_bLactam_2X_prot_DB.psq
blast_emm_Gene-nucl_DB.ndb
blast_emm_Gene-nucl_DB.nhr
blast_emm_Gene-nucl_DB.nin
blast_emm_Gene-nucl_DB.njs
blast_emm_Gene-nucl_DB.not
blast_emm_Gene-nucl_DB.nsq
blast_emm_Gene-nucl_DB.ntf
blast_emm_Gene-nucl_DB.nto
blast_frwd_primr-nucl_DB.nhr
blast_frwd_primr-nucl_DB.nin
blast_frwd_primr-nucl_DB.nsq
emm_Gene-DB_Final.fasta
emm_Gene-DB_v2_Final.

In [ ]:
# See what's actually inside — likely FASTA/allele files for pbp typing
!find GAS_Scripts_Reference/GAS_Reference_DB -type f | head -30

GAS_Scripts_Reference/GAS_Reference_DB/GAS_Res_Gene-DB_Final.fasta
GAS_Scripts_Reference/GAS_Reference_DB/GAS_Res_Gene-DB_Final.fasta.4.bt2
GAS_Scripts_Reference/GAS_Reference_DB/GAS_features_Gene-DB_Final.fasta.fai
GAS_Scripts_Reference/GAS_Reference_DB/Streptococcus_pyogenes.fasta.fai
GAS_Scripts_Reference/GAS_Reference_DB/ResFinder.fasta.4.bt2
GAS_Scripts_Reference/GAS_Reference_DB/GAS_features_Gene-DB_Final.fasta.4.bt2
GAS_Scripts_Reference/GAS_Reference_DB/GAS_features_Gene-DB_Final.fasta.3.bt2
GAS_Scripts_Reference/GAS_Reference_DB/blast_frwd_primr-nucl_DB.nsq
GAS_Scripts_Reference/GAS_Reference_DB/blast_emm_Gene-nucl_DB.nhr
GAS_Scripts_Reference/GAS_Reference_DB/Streptococcus_pyogenes.fasta
GAS_Scripts_Reference/GAS_Reference_DB/GAS_features_Gene-DB_Final.fasta
GAS_Scripts_Reference/GAS_Reference_DB/alleles_fasta
GAS_Scripts_Reference/GAS_Reference_DB/emm_Gene-DB_v2_Final.fasta
GAS_Scripts_Reference/GAS_Reference_DB/GAS_bLactam_Ref.fasta
GAS_Scripts_Reference/GAS_Reference_DB/bl

In [ ]:
import json

records = []
with open('bioproject_summary.jsonl') as f:
    for line in f:
        d = json.loads(line)
        bs = d.get('biosample', {})
        attrs = {a['name']: a['value'] for a in bs.get('attributes', [])}
        records.append({
            'accession': d.get('accession'),
            'strain': bs.get('strain'),
            'biosample': bs.get('accession'),
            'geo_loc': attrs.get('geo_loc_name'),
            'collection_date': attrs.get('collection_date'),
            'host_disease': attrs.get('host_disease'),
            'isolation_source': attrs.get('isolation_source'),
            'completeness': d.get('checkm_info', {}).get('completeness'),
            'contamination': d.get('checkm_info', {}).get('contamination'),
        })

meta_df = pd.DataFrame(records)
print(meta_df.shape)
meta_df.head()

(989, 9)


,accession,strain,biosample,geo_loc,collection_date,host_disease,isolation_source,completeness,contamination
0,GCF_053048175.1,None,None,None,None,None,None,99.32,NaN
1,GCF_053048195.1,None,None,None,None,None,None,99.16,0.14
2,GCF_053048215.1,None,None,None,None,None,None,98.14,0.72
3,GCF_053048235.1,None,None,None,None,None,None,99.43,NaN
4,GCF_053048255.1,None,None,None,None,None,None,99.02,NaN


In [ ]:
meta_df['geo_loc'].value_counts()

,count
geo_loc,


In [ ]:
!head -c 1000 GAS_Scripts_Reference/GAS_Reference_DB/GAS_bLactam_Ref.fasta
print("\n\n---ALLELE DB---\n")
!grep ">" GAS_Scripts_Reference/GAS_Reference_DB/GAS_bLactam_2X-DB_v2.faa | head -30
!echo "..."
!grep -c ">" GAS_Scripts_Reference/GAS_Reference_DB/GAS_bLactam_2X-DB_v2.faa

>GAS2X-1
GGCTCAACCATGAAAGTGATGACTCTGGCAGCGGCTATTGATGATAAAGTTTTCAACCCA
AACGAAACCTTTAGCAATGCTAATGGTTTAACAATTGCAGATGCTACTATTCAAGACTGG
TCTATTAACGAAGGCATTTCCACAGGACAGTACATGAATTATGCACAAGGGTTTGCCTTC
TCAAGTAACGTTGGGATGACTAAACTTGAACAAAAAATGGGTAATGCAAAATGGATGAAT
TATTTGACGAAGTTCCGCTTTGGTTTTCCTACTCGTTTTGGCTTAAAAGATGAAGACGCA
GGTATATTTCCTTCTGATAATATCGTGACTCAAGCTATGAGCGCTTTTGGTCAAGGGATT
TCTGTAACCCAGATTCAAATGCTTAGAGCCTTTACTGCTATTTCTAATAATGGTGAGATG
TTAGAGCCACAATTTATCAGTCAAATTTATGATCCTAACACAGCAAGTTTTAGAACGGCA
AATAAAGAAATTGTTGGAAAACCTGTATCAAAAAAAGCCGCTAGTGAAACAAGACAATAC
ATGATTGGTGTAGGAACAGACCCTGAGTTTGGAACACTCTATTCAAAAACATTTGGACCA
ATTATTAAAGTGGGTGATTTACCTGTTGCTGTTAAATCAGGAACAGCACAAATTGGTTCA
GAAGATGGAAGTGGTTATCAAGATGGTGGATTGACTAACTATGTCTATTCGGTTGTGGCA
ATGGTGCCAGCTGATAAACCAGACTTTTTGATGTATGTTACTATGACTAAACCACAACAT
TTTGGTCCCCTTTTTTGGCAAGATGTGGTTAACCCAGTATTGGAAGAAGCATACTTAATG
CAAGATACACTAACTAAGCCAGTAGTATCAGATGCTAATCGTCAAACAACTTATAAATTA
CCAAACTTTGTAGGAAAGAATCCTGGTGAGACATCAAGCGAGTTGCGTCGAAATCTTGTC
CAGCCAGTTGTCCTT

In [ ]:
!head -c 500 GAS_Scripts_Reference/GAS_Reference_DB/Gas2x270-688.tfa

>GAS2X1 GAS2X1 
PIQTFLETQMDVFQAKSNGQLASATLVNAKTGEILATTQRPTYNADTLKGLENTNYKWYS
ALHQGNFEPGSTMKVMTLAAAIDDKVFNPNETFSNANGLTIADATIQDWSINEGISTGQY
MNYAQGFAFSSNVGMTKLEQKMGNAKWMNYLTKFRFGFPTRFGLKDEDAGIFPSDNIVTQ
AMSAFGQGISVTQIQMLRAFTAISNNGEMLEPQFISQIYDPNTASFRTANKEIVGKPVSK
KAASETRQYMIGVGTDPEFGTLYSKTFGPIIKVGDLPVAVKSGTAQIGSEDGSGYQDGGL
TNYVYSVVAMVPADKPDFLMYVTMTKPQHFGPLFWQDVVNPVLEEAYLMQDTLTKPVVSD
ANRQTTYKLPNFVGKNPGETSSELRRNLVQPVVLGTGSKIKKVSHQPGQTLTENQQVLI
>GAS2X10 GAS2X10 
PIQTFLETQMDVFQAKSNGQLASATLVNAKTGEILATTQR

In [ ]:
print(meta_df['geo_loc'].value_counts(dropna=False))
print(meta_df[['accession','strain','geo_loc','collection_date']].head(10))

geo_loc
None    989
Name: count, dtype: int64
         accession strain geo_loc collection_date
0  GCF_053048175.1   None    None            None
1  GCF_053048195.1   None    None            None
2  GCF_053048215.1   None    None            None
3  GCF_053048235.1   None    None            None
4  GCF_053048255.1   None    None            None
5  GCF_053048275.1   None    None            None
6  GCF_053048295.1   None    None            None
7  GCF_053048315.1   None    None            None
8  GCF_053048335.1   None    None            None
9  GCF_053048355.1   None    None            None


In [ ]:
!grep -i -A 3 -B 3 "bLactam\|pbp2x\|GAS2X" GAS_Scripts_Reference/GAS_Res_Typer.pl | head -100

In [ ]:
records = []
with open('bioproject_summary.jsonl') as f:
    for line in f:
        d = json.loads(line)
        asm_info = d.get('assembly_info', {})
        bs = asm_info.get('biosample', {})
        attrs = {a['name']: a['value'] for a in bs.get('attributes', [])}
        records.append({
            'accession': d.get('accession'),
            'strain': bs.get('strain'),
            'biosample': bs.get('accession'),
            'geo_loc': attrs.get('geo_loc_name'),
            'collection_date': attrs.get('collection_date'),
            'host_disease': attrs.get('host_disease'),
            'isolation_source': attrs.get('isolation_source'),
            'completeness': d.get('checkm_info', {}).get('completeness'),
            'contamination': d.get('checkm_info', {}).get('contamination'),
        })

meta_df = pd.DataFrame(records)
print(meta_df.shape)
meta_df.head()

(989, 9)


,accession,strain,biosample,geo_loc,collection_date,host_disease,isolation_source,completeness,contamination
0,GCF_053048175.1,GAS-191193,SAMN52394264,USA:Georgia,2021,Pharyngitis,Oropharynx,99.32,NaN
1,GCF_053048195.1,GAS-986663,SAMN52394265,USA:Illinois,2020,Pharyngitis,Oropharynx,99.16,0.14
2,GCF_053048215.1,GAS-790155,SAMN52394266,USA:Illinois,2023,Pharyngitis,Oropharynx,98.14,0.72
3,GCF_053048235.1,GAS-650513,SAMN52394263,USA:Georgia,2020,Pharyngitis,Oropharynx,99.43,NaN
4,GCF_053048255.1,GAS-324070,SAMN52394262,USA:Illinois,2023,Pharyngitis,Oropharynx,99.02,NaN


In [ ]:
print(meta_df['geo_loc'].value_counts(dropna=False))

geo_loc
USA:Illinois     543
USA:Georgia      348
USA:Oregon        86
USA:Arizona       10
USA: Illinois      2
Name: count, dtype: int64


In [ ]:
!grep -i -A 3 -B 3 "bLactam\|pbp2x\|GAS2X" GAS_Scripts_Reference/GAS_Res_Typer.pl | head -100

In [ ]:
# Search all scripts for anything referencing the bLactam DB or 2X allele files
!grep -rli "blactam\|2x-db\|gas2x\|pbp" GAS_Scripts_Reference/*.pl GAS_Scripts_Reference/*.sh

GAS_Scripts_Reference/GAS_Target2MIC.pl
GAS_Scripts_Reference/GAS_pbpDB-Updater.sh
GAS_Scripts_Reference/StrepLab-JanOw_GAS-Typer.sh
GAS_Scripts_Reference/StrepLab-JanOw_GAS-wrapr.sh


In [ ]:
# Loosen the pattern — case-insensitive, no line restriction, just find where "2X" or "MIC" appears
!grep -in "2x\|MIC\b" GAS_Scripts_Reference/GAS_Target2MIC.pl | head -60

In [ ]:
!wc -l GAS_Scripts_Reference/GAS_Target2MIC.pl
!cat GAS_Scripts_Reference/GAS_Target2MIC.pl

214 GAS_Scripts_Reference/GAS_Target2MIC.pl
#!/bin/env perl

use strict;
use warnings;
use Data::Dumper;
#use Getopt::Long;
use Getopt::Std;
use File::Copy qw(copy);
use Env;
#use lib $ENV{MODULESHOME}."/init";
use lib "/usr/share/Modules/init/";
use perl;

###Start Doing Stuff###
my $Res_output = "RES-MIC_".$ARGV[1];
open(my $fh,'>',$Res_output) or die "Could not open file '$Res_output' $!";
#print Dumper \@ARGV;
print "Output file name is: $Res_output\n";

my %Res_hash;
my $RES_full_name = $ARGV[0];
open(MYINPUTFILE, "$RES_full_name");
while(<MYINPUTFILE>) {
    #next if $. < 2;
    my $line = $_;
    chomp($line);
    #print "$line\n";
    my @res_arr;
    @res_arr = split('\t',$line);
    $Res_hash{$res_arr[0]} = $res_arr[1];
}
close MYINPUTFILE;

my $PBP_full_name = $ARGV[2];
my $pbp_type = "NA";
open (MYINPUTFILE, "$PBP_full_name");
while (<MYINPUTFILE>) {
    next if $. < 2;
    my $line = $_;
    chomp($line);
    #print "$line\n";
    my @pbp_arr;
    @pbp_arr = split('\t',$li

In [ ]:
!grep -in "2x\|pbp" GAS_Scripts_Reference/StrepLab-JanOw_GAS-wrapr.sh | head -40

104:printf "Sample\temm_Type\tST\tgki\tgtr\tmurI\tmutS\trecP\txpt\tyqiL\tT_Type\tGroup_A\tEMM_Family\tOther_Surface_Proteins\tCapsule\tSDA1\tSLAA\tSIC\tROCA\tPNGA3\tNADase_D330G\tExotoxins\tPBP_ID\tWGS_ZOX_SIGN\tWGS_ZOX\tWGS_ZOX_SIR\tWGS_FOX_SIGN\tWGS_FOX\tWGS_FOX_SIR\tWGS_TAX_SIGN\tWGS_TAX\tWGS_TAX_SIR\tWGS_CFT_SIGN\tWGS_CFT\tWGS_CFT_SIR\tWGS_CPT_SIGN\tWGS_CPT\tWGS_CPT_SIR\tWGS_CZL_SIGN\tWGS_CZL\tWGS_CZL_SIR\tWGS_AMP_SIGN\tWGS_AMP\tWGS_AMP_SIR\tWGS_PEN_SIGN\tWGS_PEN\tWGS_PEN_SIR\tWGS_MER_SIGN\tWGS_MER\tWGS_MER_SIR\tER_CL\tWGS_ERY_SIGN\tWGS_ERY\tWGS_ERY_SIR\tWGS_CLI_SIGN\tWGS_CLI\tWGS_CLI_SIR\tWGS_LZO_SIGN\tWGS_LZO\tWGS_LZO_SIR\tWGS_SYN_SIGN\tWGS_SYN\tWGS_SYN_SIR\tWGS_ERY/CLI\tTET\tWGS_TET_SIGN\tWGS_TET\tWGS_TET_SIR\tGYRA_PARC\tWGS_CIP_SIGN\tWGS_CIP\tWGS_CIP_SIR\tWGS_LFX_SIGN\tWGS_LFX\tWGS_LFX_SIR\tOTHER\tWGS_DAP_SIGN\tWGS_DAP\tWGS_DAP_SIR\tWGS_VAN_SIGN\tWGS_VAN\tWGS_VAN_SIR\tWGS_RIF_SIGN\tWGS_RIF\tWGS_RIF_SIR\tWGS_CHL_SIGN\tWGS_CHL\tWGS_CHL_SIR\tWGS_SXT_SIGN\tWGS_SXT\tWGS_SXT_SIR\tCon

In [ ]:
!grep -in "2x\|pbp\|blast" GAS_Scripts_Reference/GAS_Res_Typer.pl | head -60

In [ ]:
!grep -in "2x\|pbp\|blast" GAS_Scripts_Reference/GAS_Res_Typer.pl

In [ ]:
!cat GAS_Scripts_Reference/GAS_pbpDB-Updater.sh

#!/bin/bash -l

. /usr/share/Modules/init/bash
module load Python/2.7
module load ncbi-blast+/2.2.29

#Comment blah...#

while getopts :r:u:t: option
do
    case $option in
        r) refSeq_path=$OPTARG;;
        u) update_PBP=$OPTARG;;
        t) table_out=$OPTARG;;
    esac
done

###Check if the PBP reference paths exist and are correct###
if [[ -z "$refSeq_path" ]]
then
    echo "No PBP reference database path argument given."
    exit 1
else
    refSeq_p2=$(echo $refSeq_path | sed 's/\/$//g')
    refSeq="${refSeq_p2}"
    echo "refSeq: $refSeq"
fi

#Ref_1A=$(ls $refSeq/*1A*.faa)
Ref_2X=$(ls $refSeq/*2X*.faa)

#if [[ -e "$Ref_1A" && -e "$Ref_2X" ]]
if [[ -e "$Ref_2X" ]]
then
    echo "The PBP 2X reference database is: $Ref_2X"
else
    echo "The PBP 2X reference database is not in the correct format or doesn't exist."
    echo "Make sure it's in the $refSeq directory and has the correct naming format."
    echo "Specifically, the PBP sequence file must contain '2X' and have a '.faa

In [ ]:
!grep -in "PBP_ID\|newPBP\|blastp\|blastn" GAS_Scripts_Reference/StrepLab-JanOw_GAS-wrapr.sh

104:printf "Sample\temm_Type\tST\tgki\tgtr\tmurI\tmutS\trecP\txpt\tyqiL\tT_Type\tGroup_A\tEMM_Family\tOther_Surface_Proteins\tCapsule\tSDA1\tSLAA\tSIC\tROCA\tPNGA3\tNADase_D330G\tExotoxins\tPBP_ID\tWGS_ZOX_SIGN\tWGS_ZOX\tWGS_ZOX_SIR\tWGS_FOX_SIGN\tWGS_FOX\tWGS_FOX_SIR\tWGS_TAX_SIGN\tWGS_TAX\tWGS_TAX_SIR\tWGS_CFT_SIGN\tWGS_CFT\tWGS_CFT_SIR\tWGS_CPT_SIGN\tWGS_CPT\tWGS_CPT_SIR\tWGS_CZL_SIGN\tWGS_CZL\tWGS_CZL_SIR\tWGS_AMP_SIGN\tWGS_AMP\tWGS_AMP_SIR\tWGS_PEN_SIGN\tWGS_PEN\tWGS_PEN_SIR\tWGS_MER_SIGN\tWGS_MER\tWGS_MER_SIR\tER_CL\tWGS_ERY_SIGN\tWGS_ERY\tWGS_ERY_SIR\tWGS_CLI_SIGN\tWGS_CLI\tWGS_CLI_SIR\tWGS_LZO_SIGN\tWGS_LZO\tWGS_LZO_SIR\tWGS_SYN_SIGN\tWGS_SYN\tWGS_SYN_SIR\tWGS_ERY/CLI\tTET\tWGS_TET_SIGN\tWGS_TET\tWGS_TET_SIR\tGYRA_PARC\tWGS_CIP_SIGN\tWGS_CIP\tWGS_CIP_SIR\tWGS_LFX_SIGN\tWGS_LFX\tWGS_LFX_SIR\tOTHER\tWGS_DAP_SIGN\tWGS_DAP\tWGS_DAP_SIR\tWGS_VAN_SIGN\tWGS_VAN\tWGS_VAN_SIR\tWGS_RIF_SIGN\tWGS_RIF\tWGS_RIF_SIR\tWGS_CHL_SIGN\tWGS_CHL\tWGS_CHL_SIR\tWGS_SXT_SIGN\tWGS_SXT\tWGS_SXT_SIR\tCon

In [ ]:
!apt-get install -y ncbi-blast+ -qq
!blastp -version

Selecting previously unselected package ncbi-data.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../ncbi-data_6.1.20170106+dfsg1-9_all.deb ...
Unpacking ncbi-data (6.1.20170106+dfsg1-9) ...
Selecting previously unselected package ncbi-blast+.
Preparing to unpack .../ncbi-blast+_2.12.0+ds-3build1_amd64.deb ...
Unpacking ncbi-blast+ (2.12.0+ds-3build1) ...
Setting up ncbi-data (6.1.20170106+dfsg1-9) ...
Setting up ncbi-blast+ (2.12.0+ds-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
Processing triggers for hicolor-icon-theme (0.17-2) ...
blastp: 2.12.0+
 Package: blast 2.12.0, build Mar  8 2022 16:19:08


In [ ]:
# Pick 3 accessions from meta_df to pilot the pipeline
test_accessions = meta_df['accession'].iloc[:3].tolist()
print(test_accessions)

for acc in test_accessions:
    !./datasets download genome accession {acc} --include protein --filename {acc}.zip
    !unzip -o -q {acc}.zip -d {acc}_data

['GCF_053048175.1', 'GCF_053048195.1', 'GCF_053048215.1']
Downloading: GCF_053048175.1.zip    2.68kB 16.7MB/s
Downloading: GCF_053048175.1.zip    2.68kB 16.7MB/s
Downloading: GCF_053048175.1.zip    32kB 1.3MB/s
Downloading: GCF_053048175.1.zip    32kB 1.3MB/s
Downloading: GCF_053048175.1.zip    32kB 1.3MB/s
Downloading: GCF_053048175.1.zip    65.5kB 1.21MB/s
Downloading: GCF_053048175.1.zip    65.5kB 1.21MB/s
Downloading: GCF_053048175.1.zip    197kB 2.59MB/s
Downloading: GCF_053048175.1.zip    197kB 2.59MB/s
Downloading: GCF_053048175.1.zip    351kB done
Validating package files [==================>-----------------------------]  40% 2/5
Downloading: GCF_053048175.1.zip    351kB valid data package
Validating package files [================================================] 100% 5/5
Downloading: GCF_053048195.1.zip    847B 8.81MB/s
Downloading: GCF_053048195.1.zip    847B 8.81MB/s
Downloading: GCF_053048195.1.zip    2.61kB 108kB/s
Downloading: GCF_053048195.1.zip    2.61kB 108kB/s
Downl

In [ ]:
import glob

for acc in test_accessions:
    protein_files = glob.glob(f'{acc}_data/ncbi_dataset/data/*/protein.faa')
    print(f"{acc}: {protein_files}")
    if protein_files:
        !grep -c ">" {protein_files[0]}

GCF_053048175.1: ['GCF_053048175.1_data/ncbi_dataset/data/GCF_053048175.1/protein.faa']
1700
GCF_053048195.1: ['GCF_053048195.1_data/ncbi_dataset/data/GCF_053048195.1/protein.faa']
1890
GCF_053048215.1: ['GCF_053048215.1_data/ncbi_dataset/data/GCF_053048215.1/protein.faa']
1704


In [ ]:
# Step A: make a BLAST protein DB for each isolate's full proteome
for acc in test_accessions:
    protein_file = glob.glob(f'{acc}_data/ncbi_dataset/data/*/protein.faa')[0]
    !makeblastdb -in {protein_file} -dbtype prot -out {acc}_protdb -logfile /dev/null

# Step B: BLAST the reference pbp2x (translated) against each isolate's proteome
# to find which protein in THAT isolate is the pbp2x homolog
!blastx -query GAS_Scripts_Reference/GAS_Reference_DB/GAS_bLactam_Ref.fasta \
    -db {test_accessions[0]}_protdb -outfmt 6 -max_target_seqs 1 > {test_accessions[0]}_pbp2x_hit.tsv

In [ ]:
for acc in test_accessions:
    !blastx -query GAS_Scripts_Reference/GAS_Reference_DB/GAS_bLactam_Ref.fasta \
        -db {acc}_protdb -outfmt 6 -max_target_seqs 1 -evalue 1e-10 > {acc}_pbp2x_hit.tsv
    print(f"--- {acc} ---")
    !cat {acc}_pbp2x_hit.tsv

--- GCF_053048175.1 ---
GAS2X-1	WP_060388530.1	99.713	349	1	0	1	1047	339	687	0.0	719
--- GCF_053048195.1 ---
GAS2X-1	WP_447278074.1	99.140	349	3	0	1	1047	339	687	0.0	716
--- GCF_053048215.1 ---
GAS2X-1	WP_219095966.1	99.713	349	1	0	1	1047	339	687	0.0	719


In [ ]:
!pip install Bio
from Bio import SeqIO

hit_ids = {
    'GCF_053048175.1': 'WP_060388530.1',
    'GCF_053048195.1': 'WP_447278074.1',
    'GCF_053048215.1': 'WP_219095966.1',
}

extracted = {}
for acc, hit_id in hit_ids.items():
    protein_file = glob.glob(f'{acc}_data/ncbi_dataset/data/*/protein.faa')[0]
    for record in SeqIO.parse(protein_file, 'fasta'):
        if record.id == hit_id:
            extracted[acc] = str(record.seq)
            break
    print(acc, hit_id, len(extracted.get(acc, '')), 'aa')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.4/321.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.1 MB/s eta 0:00:00
GCF_053048175.1 WP_060388530.1 751 aa
GCF_053048195.1 WP_447278074.1 751 aa
GCF_053048215.1 WP_219095966.1 751 aa


In [ ]:
with open('test_pbp2x_query.fasta', 'w') as f:
    for acc, seq in extracted.items():
        f.write(f'>{acc}\n{seq}\n')

!makeblastdb -in GAS_Scripts_Reference/GAS_Reference_DB/GAS_bLactam_2X-DB_v2.faa -dbtype prot -out allele_db -logfile /dev/null

!blastp -query test_pbp2x_query.fasta -db allele_db -outfmt "6 qseqid sseqid pident length mismatch evalue bitscore" -max_target_seqs 3 -evalue 1e-10 > allele_hits.tsv
!cat allele_hits.tsv

GCF_053048175.1	10||GAS_2X	100.000	419	0	0.0	867
GCF_053048175.1	48||GAS_2X	99.761	419	1	0.0	865
GCF_053048175.1	25||GAS_2X	99.761	419	1	0.0	865
GCF_053048195.1	46||GAS_2X	100.000	419	0	0.0	867
GCF_053048195.1	9||GAS_2X	99.761	419	1	0.0	864
GCF_053048195.1	42||GAS_2X	99.523	419	2	0.0	863
GCF_053048215.1	16||GAS_2X	99.761	419	1	0.0	865
GCF_053048215.1	20||GAS_2X	99.761	419	1	0.0	865
GCF_053048215.1	28||GAS_2X	99.761	419	1	0.0	864


In [ ]:
!cat GAS_Scripts_Reference/GAS_Target2MIC.pl | grep -n -B5 -A40 "sub.*[Mm]ic\|safe\|Flag\|RBLS" | head -150

68-	$Out_hash{TET} = "$Res_hash{TET},>=,8,R";
69-    }
70-}
71-
72-###PBP Category###
73:$drug{ZOX} = "Flag,Flag,Flag";
74:$drug{FOX} = "Flag,Flag,Flag";
75:$drug{TAX} = "Flag,Flag,Flag";
76:$drug{CZL} = "Flag,Flag,Flag";
77:$drug{CFT} = "Flag,Flag,Flag";
78:$drug{CPT} = "Flag,Flag,Flag";
79:$drug{AMP} = "Flag,Flag,Flag";
80:$drug{PEN} = "Flag,Flag,Flag";
81:$drug{MER} = "Flag,Flag,Flag";
82-#if ($pbp_type <= 18 && $pbp_type != [11|17] && $pbp_type !~ /[A-Za-z]/) {
83-if ((($pbp_type <= 20 && $pbp_type !~ /^(11|17)$/) || $pbp_type =~ /^(51|58|73)$/) && $pbp_type !~ /[A-Za-z]/) {
84-    $drug{ZOX} = "<=,0.12,U";
85-    $drug{FOX} = "<=,2,U";
86-    $drug{TAX} = "<=,0.06,S";
87-    $drug{CFT} = "<=,0.06,S";
88-    $drug{CPT} = "<=,0.06,S";
89-    $drug{CZL} = "<=,0.5,U";
90-    $drug{AMP} = "<=,0.06,S";
91-    $drug{PEN} = "<=,0.06,S";
92-    $drug{MER} = "<=,0.06,S";
93-}
94-print "PBP,$drug{ZOX},$drug{FOX},$drug{TAX},$drug{CFT},$drug{CPT},$drug{CZL},$drug{AMP},$drug{PEN},$drug{MER}\n";

In [ ]:
!grep -n "^\s*[0-9]" GAS_Scripts_Reference/GAS_Target2MIC.pl | grep -i "flag\|safe\|susc\|resist"

In [ ]:
import glob, subprocess

def get_pbp2x_allele(acc):
    protein_file_list = glob.glob(f'{acc}_data/ncbi_dataset/data/*/protein.faa')
    if not protein_file_list:
        return None, None
    protein_file = protein_file_list[0]

    !makeblastdb -in {protein_file} -dbtype prot -out {acc}_protdb -logfile /dev/null
    !blastx -query GAS_Scripts_Reference/GAS_Reference_DB/GAS_bLactam_Ref.fasta \
        -db {acc}_protdb -outfmt 6 -max_target_seqs 1 -evalue 1e-10 > {acc}_hit.tsv

    with open(f'{acc}_hit.tsv') as f:
        line = f.readline().strip()
    if not line:
        return None, None
    hit_id = line.split('\t')[1]

    from Bio import SeqIO
    seq = None
    for record in SeqIO.parse(protein_file, 'fasta'):
        if record.id == hit_id:
            seq = str(record.seq)
            break
    if seq is None:
        return hit_id, None

    with open(f'{acc}_query.fasta', 'w') as f:
        f.write(f'>{acc}\n{seq}\n')
    !blastp -query {acc}_query.fasta -db allele_db -outfmt "6 sseqid pident" -max_target_seqs 1 -evalue 1e-10 > {acc}_allele.tsv
    with open(f'{acc}_allele.tsv') as f:
        aline = f.readline().strip()
    if not aline:
        return hit_id, None
    allele = aline.split('\t')[0].split('||')[0]
    return hit_id, allele

# test on the same 3 first to sanity-check the function matches Cells 17-22 manual results
for acc in test_accessions:
    print(acc, get_pbp2x_allele(acc))

GCF_053048175.1 ('WP_060388530.1', '10')
GCF_053048195.1 ('WP_447278074.1', '46')
GCF_053048215.1 ('WP_219095966.1', '16')


In [ ]:
import time

all_accessions = meta_df['accession'].tolist()
print(f"Total accessions: {len(all_accessions)}")

batch_size = 50
for i in range(0, len(all_accessions), batch_size):
    batch = all_accessions[i:i+batch_size]
    acc_str = ' '.join(batch)
    print(f"Downloading batch {i}-{i+len(batch)}...")
    !./datasets download genome accession {acc_str} --include protein --filename batch_{i}.zip
    !unzip -o -q batch_{i}.zip -d batch_{i}_data
    time.sleep(1)  # small pause to be polite to NCBI's servers

Streaming output truncated to the last 5000 lines.
Downloading: batch_550.zip    11.7MB 3.73MB/s
Downloading: batch_550.zip    11.7MB 3.73MB/s
Downloading: batch_550.zip    11.9MB 3.74MB/s
Downloading: batch_550.zip    12.1MB 3.79MB/s
Downloading: batch_550.zip    12.1MB 3.79MB/s
Downloading: batch_550.zip    12.1MB 3.76MB/s
Downloading: batch_550.zip    12.4MB 3.85MB/s
Downloading: batch_550.zip    12.4MB 3.85MB/s
Downloading: batch_550.zip    12.4MB 3.85MB/s
Downloading: batch_550.zip    12.4MB 3.85MB/s
Downloading: batch_550.zip    12.4MB 3.85MB/s
Downloading: batch_550.zip    12.4MB 3.79MB/s
Downloading: batch_550.zip    12.6MB 3.82MB/s
Downloading: batch_550.zip    12.7MB 3.87MB/s
Downloading: batch_550.zip    12.8MB 3.88MB/s
Downloading: batch_550.zip    13.1MB 3.95MB/s
Downloading: batch_550.zip    13.1MB 3.95MB/s
Downloading: batch_550.zip    13.1MB 3.95MB/s
Downloading: batch_550.zip    13.1MB 3.95MB/s
Downloading: batch_550.zip    13.1MB 3.95MB/s
Downloading: batch_550.zip   

In [ ]:
protein_path_map = {}
for i in range(0, len(all_accessions), batch_size):
    paths = glob.glob(f'batch_{i}_data/ncbi_dataset/data/*/protein.faa')
    for p in paths:
        acc_from_path = p.split('/')[-2]
        protein_path_map[acc_from_path] = p

print(f"Found protein.faa for {len(protein_path_map)} / {len(all_accessions)} accessions")
missing = set(all_accessions) - set(protein_path_map.keys())
print(f"Missing: {len(missing)}")
list(missing)[:10]

Found protein.faa for 989 / 989 accessions
Missing: 0


[]

In [ ]:
import os, csv

results_file = 'pbp2x_alleles.csv'
already_done = set()
if os.path.exists(results_file):
    with open(results_file) as f:
        already_done = {row.split(',')[0] for row in f.readlines()[1:]}
else:
    with open(results_file, 'w') as f:
        f.write('accession,hit_protein_id,allele\n')

def get_pbp2x_allele_v2(acc, protein_file):
    !makeblastdb -in {protein_file} -dbtype prot -out {acc}_protdb -logfile /dev/null
    !blastx -query GAS_Scripts_Reference/GAS_Reference_DB/GAS_bLactam_Ref.fasta \
        -db {acc}_protdb -outfmt 6 -max_target_seqs 1 -evalue 1e-10 2>/dev/null > {acc}_hit.tsv
    with open(f'{acc}_hit.tsv') as f:
        line = f.readline().strip()
    if not line:
        return None, None
    hit_id = line.split('\t')[1]

    from Bio import SeqIO
    seq = None
    for record in SeqIO.parse(protein_file, 'fasta'):
        if record.id == hit_id:
            seq = str(record.seq)
            break
    if seq is None:
        return hit_id, None

    with open(f'{acc}_query.fasta', 'w') as f:
        f.write(f'>{acc}\n{seq}\n')
    !blastp -query {acc}_query.fasta -db allele_db -outfmt "6 sseqid pident" -max_target_seqs 1 -evalue 1e-10 2>/dev/null > {acc}_allele.tsv
    with open(f'{acc}_allele.tsv') as f:
        aline = f.readline().strip()
    if not aline:
        return hit_id, None
    allele = aline.split('\t')[0].split('||')[0]

    # cleanup intermediate files to save disk space
    for ext in ['_protdb.phr','_protdb.pin','_protdb.psq','_hit.tsv','_query.fasta','_allele.tsv']:
        try: os.remove(f'{acc}{ext}')
        except: pass

    return hit_id, allele

for idx, acc in enumerate(all_accessions):
    if acc in already_done:
        continue
    if acc not in protein_path_map:
        continue
    hit_id, allele = get_pbp2x_allele_v2(acc, protein_path_map[acc])
    with open(results_file, 'a') as f:
        f.write(f'{acc},{hit_id or ""},{allele or ""}\n')
    if idx % 25 == 0:
        print(f"{idx}/{len(all_accessions)} done...")

print("Done.")

0/989 done...
25/989 done...
50/989 done...
75/989 done...
100/989 done...
125/989 done...
150/989 done...
175/989 done...
200/989 done...
225/989 done...
250/989 done...
275/989 done...
300/989 done...
325/989 done...
350/989 done...
375/989 done...
400/989 done...
425/989 done...
450/989 done...
475/989 done...
500/989 done...
525/989 done...
550/989 done...
575/989 done...
600/989 done...
625/989 done...
650/989 done...
675/989 done...
700/989 done...
725/989 done...
750/989 done...
775/989 done...
800/989 done...
825/989 done...
850/989 done...
875/989 done...
900/989 done...
925/989 done...
950/989 done...
975/989 done...
Done.


In [ ]:
df = pd.read_csv('pbp2x_alleles.csv')
print(df.shape)
print(df['allele'].value_counts(dropna=False).head(20))
print(df[df['allele'].isna()].shape)

(989, 3)
allele
1.0     708
4.0     111
3.0      60
8.0      48
10.0     34
16.0     11
24.0      4
46.0      2
NaN       2
20.0      2
31.0      2
2.0       1
42.0      1
53.0      1
51.0      1
45.0      1
Name: count, dtype: int64
(2, 3)


In [ ]:
# Re-run extraction but this time keep the sequence, not just the allele number
import csv

seq_records = []
allele_df = pd.read_csv('pbp2x_alleles.csv')

with open('pbp2x_sequences.fasta', 'w') as fasta_out:
    for idx, row in allele_df.iterrows():
        acc = row['accession']
        hit_id = row['hit_protein_id']
        if pd.isna(hit_id) or not hit_id:
            continue
        protein_file = protein_path_map.get(acc)
        if not protein_file:
            continue
        for record in SeqIO.parse(protein_file, 'fasta'):
            if record.id == hit_id:
                fasta_out.write(f'>{acc}|allele_{row["allele"]}\n{str(record.seq)}\n')
                break
        if idx % 100 == 0:
            print(f"{idx}/{len(allele_df)} sequences written...")

print("Done.")
!grep -c ">" pbp2x_sequences.fasta

0/989 sequences written...
100/989 sequences written...
200/989 sequences written...
300/989 sequences written...
400/989 sequences written...
500/989 sequences written...
600/989 sequences written...
700/989 sequences written...
800/989 sequences written...
900/989 sequences written...
Done.
987


In [ ]:
# Part A: which 2 isolates failed, and why
nan_rows = allele_df[allele_df['allele'].isna()]
print(nan_rows)

for acc in nan_rows['accession']:
    hit_id = allele_df.loc[allele_df['accession'] == acc, 'hit_protein_id'].values[0]
    print(f"{acc}: hit_protein_id = {hit_id}")
    # if hit_id itself is missing/NaN, the blastx step found nothing — likely assembly/contig issue
    # if hit_id exists but allele is NaN, the blastp step against allele_db failed — worth re-running manually

          accession hit_protein_id  allele
12  GCF_053048415.1            NaN     NaN
55  GCA_053048415.1            NaN     NaN
GCF_053048415.1: hit_protein_id = nan
GCA_053048415.1: hit_protein_id = nan


In [ ]:
# Part B: pull sequences for the flagged (non-safe) alleles for a manual look
flagged_alleles = ['24', '31', '42', '45', '46', '53']
flagged_accessions = allele_df[allele_df['allele'].astype(str).isin(flagged_alleles)]['accession'].tolist()
print(f"{len(flagged_accessions)} flagged isolates: {flagged_accessions}")

# pull their sequences from the FASTA we just wrote
from Bio import SeqIO
for record in SeqIO.parse('pbp2x_sequences.fasta', 'fasta'):
    acc = record.id.split('|')[0]
    if acc in flagged_accessions:
        print(f">{record.id}")
        print(str(record.seq)[:100], '...')  # just the start, full seq is in the file

0 flagged isolates: []


In [ ]:
# how many proteins does this isolate even have, and is protein.faa non-empty/valid?
for acc in ['GCF_053048415.1', 'GCA_053048415.1']:
    pf = protein_path_map.get(acc)
    print(acc, pf)
    if pf:
        !grep -c ">" {pf}

GCF_053048415.1 batch_0_data/ncbi_dataset/data/GCF_053048415.1/protein.faa
1677
GCA_053048415.1 batch_50_data/ncbi_dataset/data/GCA_053048415.1/protein.faa
1708


In [ ]:
allele_df_clean = allele_df[~allele_df['accession'].isin(['GCF_053048415.1', 'GCA_053048415.1'])].copy()
print(allele_df_clean.shape)  # should be (987, 3)
allele_df_clean.to_csv('pbp2x_alleles_clean.csv', index=False)

(987, 3)


In [ ]:
flagged_alleles = ['24', '31', '42', '45', '46', '53']
flagged_accessions = allele_df_clean[
    allele_df_clean['allele'].dropna().astype(int).astype(str).isin(flagged_alleles)
]['accession'].tolist()
print(f"{len(flagged_accessions)} flagged isolates: {flagged_accessions}")

from Bio import SeqIO
for record in SeqIO.parse('pbp2x_sequences.fasta', 'fasta'):
    acc = record.id.split('|')[0]
    if acc in flagged_accessions:
        print(f">{record.id}")
        print(str(record.seq)[:100], '...')

11 flagged isolates: ['GCF_053048195.1', 'GCF_053049015.1', 'GCA_053048195.1', 'GCA_053049015.1', 'GCA_057689235.1', 'GCA_057690075.1', 'GCA_057701215.1', 'GCA_057701815.1', 'GCA_057777265.1', 'GCA_057779485.1', 'GCA_057779585.1']
>GCF_053048195.1|allele_46.0
MKKWQKYVLDYVVRDRRTPVENRVRVGQNMMLLTIFIFFIFIINFMIIIGTDQKFGVSLSEGAKKVYQETVTIQAKRGTIYDRNGTAIAVDSTTYSIYAI ...
>GCF_053049015.1|allele_24.0
MKKWQKYVLDYVVRDRRTPVENRVRVGQNMMLLTIFIFLIFIINFMIIIGTDQKFGVSLSEGAKKVYQETVTIQAKRGTIYDRNGTAIAVDSTTYSIYAI ...
>GCA_053048195.1|allele_46.0
MKKWQKYVLDYVVRDRRTPVENRVRVGQNMMLLTIFIFFIFIINFMIIIGTDQKFGVSLSEGAKKVYQETVTIQAKRGTIYDRNGTAIAVDSTTYSIYAI ...
>GCA_053049015.1|allele_24.0
MKKWQKYVLDYVVRDRRTPVENRVRVGQNMMLLTIFIFLIFIINFMIIIGTDQKFGVSLSEGAKKVYQETVTIQAKRGTIYDRNGTAIAVDSTTYSIYAI ...
>GCA_057689235.1|allele_42.0
MKKWQKYVLDYVVRDRRTPVENRVRVGQNMMLLTIFIFFIFIINFMIIIGTDQKFGVSLSEGAKKVYQETVTIQAKRGTIYDRNGTAIAVDSTTYSIYAI ...
>GCA_057690075.1|allele_53.0
MKKWQKYVLDYVVRDRRTPVENRIRVGQNMMLLTIFIFFIFIINFMIIIGTDQKFGVSLSEGAKKVYQET

In [ ]:
!zip -r day1_outputs.zip pbp2x_alleles_clean.csv pbp2x_sequences.fasta bioproject_summary.jsonl GAS_Scripts_Reference/GAS_Reference_DB/GAS_bLactam_Ref.fasta GAS_Scripts_Reference/GAS_Reference_DB/GAS_bLactam_2X-DB_v2.faa

from google.colab import files
files.download('day1_outputs.zip')

  adding: pbp2x_alleles_clean.csv (deflated 78%)
  adding: pbp2x_sequences.fasta (deflated 99%)
  adding: bioproject_summary.jsonl (deflated 95%)
  adding: GAS_Scripts_Reference/GAS_Reference_DB/GAS_bLactam_Ref.fasta (deflated 64%)
  adding: GAS_Scripts_Reference/GAS_Reference_DB/GAS_bLactam_2X-DB_v2.faa (deflated 97%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>